In [1]:
"""
TODO Create for loop version for multiple target date

Create folder in tarderquint gmail drive and put the result there.
"""


import pandas as pd
from datetime import datetime


def get_backtest_bounds(target_date: str, last_n_days: int):
    target_dt = pd.to_datetime(target_date)
    end_date_dt = target_dt - pd.Timedelta(days=1)
    start_date_dt = end_date_dt - pd.Timedelta(days=last_n_days)
    return start_date_dt.strftime('%Y-%m-%d'), end_date_dt.strftime('%Y-%m-%d')

def get_backtest_bounds(target_date: str, last_n_days: int):
    target_dt = pd.to_datetime(target_date)
    end_date_dt = target_dt - pd.offsets.BusinessDay(1)
    start_date_dt = end_date_dt - pd.offsets.BusinessDay(last_n_days)
    return start_date_dt.strftime('%Y-%m-%d'), end_date_dt.strftime('%Y-%m-%d')

# P.S. assume the data is queried
target_date = "2026-05-08"
last_n_days = 30
start_date, end_date = get_backtest_bounds(target_date, last_n_days)

In [10]:
start_date, end_date

('2026-03-26', '2026-05-07')

In [2]:
# TO CHANGE
# use your save_dir
save_dir = "/Users/livardywufianto/Projects/Trading/data/minute_aggs"

In [3]:
ticker_list = [
    "AAPL",
    "MU",
]

import pandas as pd
filter_df = pd.read_csv("/Users/livardywufianto/Projects/Trading/data/stock_filter.csv")
ticker_list = filter_df["Ticker"].values.tolist()

# Load Data

In [4]:
import pandas as pd
from typing import List
import pandas as pd

def generate_csv_filepaths(start_date, end_date, save_dir):
    date_series = pd.date_range(start=start_date, end=end_date)    
    keys = []
    base_path = save_dir
    
    for dt in date_series:
        full_date = dt.strftime('%Y-%m-%d')
        key = f"{base_path}/{full_date}.csv.gz"
        keys.append(key)        
    return keys

def generate_csv_filepath(date_str, save_dir):
    dt = pd.to_datetime(date_str)
    base_path = save_dir
    full_date = dt.strftime('%Y-%m-%d')
    key = f"{base_path}/{full_date}.csv.gz"
    return key


def convert_timestamp_to_us_datetime(timestamp: int, unit="ms"):
    """
    Converts a millisecond timestamp to US/Eastern time.

    NYSE/NASDAQ are located at US/Eastern
    """
    return pd.to_datetime(
        timestamp, unit=unit, utc=True
    ).tz_convert('US/Eastern')      
    
def load_csv_by_chunking(
    filepath: str, ticker_list: List[str],
    chunk_size = 100000
):

    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        filtered_chunk = chunk[chunk["ticker"].isin(ticker_list)]
        chunks.append(filtered_chunk)

    df = pd.concat(chunks)
    return df

In [5]:
csv_filepaths = generate_csv_filepaths(start_date, end_date, save_dir)

In [6]:
import os

df_ls = []

for csv_filepath in csv_filepaths:
    try:
        # TO CHANGE
        # load_csv_by_chunking - use pd.read_csv()
        df_ls.append(
            load_csv_by_chunking(csv_filepath, ticker_list)
        )
    except FileNotFoundError:
        print(f"Error: The file at {csv_filepath} was not found. Skipping...")
    except Exception as e:
        print(f"An unexpected error occurred with {os.path.basename(csv_filepath)}: {e}")
        
df = pd.concat(df_ls).reset_index(drop=True)

# TO CHANGE
# load_csv_by_chunking - use pd.read_csv()
target_df = load_csv_by_chunking(
    generate_csv_filepath(target_date, save_dir), ticker_list
)

Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-03-28.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-03-29.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-04-03.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-04-04.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-04-05.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-04-11.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-04-12.csv.gz was not found. Skipping...
Error: The file at /Users/livardywufianto/Projects/Trading/data/minute_aggs/2026-04-18.csv.gz was not found. Skipping...
Error: The file at /Users/livard

In [7]:
import numpy as np

def aggregate_to_hour(df):
    df["us_datetime"] = df["window_start"].apply(
        lambda x: convert_timestamp_to_us_datetime(x, unit="ns"))
    df["us_date"] = df["us_datetime"].dt.date
    df["hour"] = df["us_datetime"].dt.hour
    agg_logic = {
        "open": "first",    # The first price of the hour
        "high": "max",      # The highest price in that hour
        "low": "min",       # The lowest price in that hour
        "close": "last",    # The final price of the hour
        "volume": "sum",    # Total volume traded in that hour
        "transactions": "sum" # Total count of trades in that hour
    }
    
    hour_df = df.groupby(
        ["ticker", "us_date", "hour"]
    ).agg(agg_logic).reset_index()
    return hour_df



# Volume Check

In [8]:
apply_log_norm = False
threshold = 2

hour_df = aggregate_to_hour(df)
target_hour_df = aggregate_to_hour(target_df)

if apply_log_norm:
    hour_df["volume"] = np.log1p(hour_df["volume"])
    target_hour_df["volume"] = np.log1p(target_hour_df["volume"])
    
volume_df = hour_df.groupby(["ticker", "hour"]).agg({
    "volume": ["mean", "std"]
})
volume_df.columns = [f"{col}_{stat}" for col, stat in volume_df.columns]
volume_df = volume_df.reset_index()

merge_df = pd.merge(target_hour_df, volume_df, on=["ticker", "hour"])
merge_df["z_score"] = (merge_df["volume"] - merge_df["volume_mean"]) / merge_df["volume_std"]
anomaly_df = merge_df[merge_df["z_score"].abs() > threshold]

filename = f"volume_check_log_norm={apply_log_norm}_threshold={threshold}.csv"
save_filepath = f"/Users/livardywufianto/Projects/Trading/data/outputs/{filename}"
anomaly_df.to_csv(save_filepath, index=False, header=True)

# Log Volume

In [9]:
apply_log_norm = True
threshold = 2

hour_df = aggregate_to_hour(df)
target_hour_df = aggregate_to_hour(target_df)
if apply_log_norm:
    hour_df["volume"] = np.log1p(hour_df["volume"])
    target_hour_df["volume"] = np.log1p(target_hour_df["volume"])
    
volume_df = hour_df.groupby(["ticker", "hour"]).agg({
    "volume": ["mean", "std"]
})
volume_df.columns = [f"{col}_{stat}" for col, stat in volume_df.columns]
volume_df = volume_df.reset_index()

merge_df = pd.merge(target_hour_df, volume_df, on=["ticker", "hour"])
merge_df["z_score"] = (merge_df["volume"] - merge_df["volume_mean"]) / merge_df["volume_std"]
anomaly_df = merge_df[merge_df["z_score"].abs() > threshold]

filename = f"volume_check_log_norm={apply_log_norm}_threshold={threshold}.csv"
save_filepath = f"/Users/livardywufianto/Projects/Trading/data/outputs/{filename}"
anomaly_df.to_csv(save_filepath, index=False, header=True)

In [13]:
anomaly_df

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
19,AA,2026-05-08,16,63.180,63.7296,62.9367,63.190,13.043329,63,10.841868,1.065002,2.067096
21,AA,2026-05-08,19,63.350,63.3500,63.3500,63.350,4.615121,1,7.698936,1.407199,-2.191456
30,AAOI,2026-05-08,4,153.510,172.7000,153.0100,168.500,12.837078,9102,11.235066,0.578382,2.769813
31,AAOI,2026-05-08,5,168.310,168.5900,163.3400,164.480,11.547815,2223,9.918582,0.649632,2.507930
33,AAOI,2026-05-08,7,162.240,171.9000,160.5200,168.500,12.430417,4994,10.993010,0.615715,2.334534
...,...,...,...,...,...,...,...,...,...,...,...,...
13139,ZTS,2026-05-08,11,83.520,83.5700,82.0500,82.620,14.491615,27914,12.738361,0.660166,2.655778
13140,ZTS,2026-05-08,12,82.590,83.5000,82.0000,82.217,13.994746,18306,12.462910,0.595277,2.573315
13141,ZTS,2026-05-08,13,82.205,82.3299,81.1000,82.215,14.229279,21818,12.468965,0.521164,3.377655
13142,ZTS,2026-05-08,14,82.215,82.2850,81.4000,81.955,14.479754,23109,12.534985,0.590282,3.294643
